In [1]:
!hostname

node010


In [1]:
import urllib3
from io import StringIO
import os
import glob
import matplotlib.pyplot as plt
import neuroboros as nb
import numpy as np
import pandas as pd
from scipy.stats import gamma, zscore
from scipy.ndimage import convolve1d

In [3]:
with np.load('./NEP/GLM/prosem_contrasts/3102_wd_02.npz') as data:
    print("beta shape:", data['beta'].shape)
    print("ts shape:", data['ts'].shape)
    print("R2s shape:", data['R2s'].shape)

beta shape: (4, 20484)
ts shape: (4, 20484)
R2s shape: (20484,)


In [26]:
import os
import numpy as np

def compute_condition_correlation_matrices(data_folder, output_dir='./subject_matrices'):
    """
    Loads (6, 20484) arrays for each run, computes a 4x4 correlation matrix 
    for each of the 6 conditions, and saves a .npy file per subject.
    """
    os.makedirs(output_dir, exist_ok=True)
    
    # 1. Define Groups
    native = list(range(3102, 3117)) 
    learner = list(range(3202, 3217))
    subjects = native + learner
    
    # 2. Define Conditions and Runs
    conditions = [
        'positive_positive', 
        'positive_negative', 
        'positive_neutral', 
        'negative_negative', 
        'negative_positive', 
        'negative_neutral'
    ]
    
    run_configs = [
        ('wd', '01'), 
        ('wd', '02'), 
        ('sen', '01'), 
        ('sen', '02')
    ]
    
    print(f"Scanning folder: {data_folder}")
    
    for sub in subjects:
        # Determine group for metadata
        group_name = 'native' if sub in native else 'learner'
        
        sub_data_ts = []
        sub_data_betas = []
        files_missing = False
        
        # Load all 4 runs for this subject
        for task, run in run_configs:
            # Note: Assuming the files are still .npz as in the previous script.
            # If they are raw .npy files without 'ts'/'betas' keys, let me know!
            filepath = os.path.join(data_folder, f"{sub}_{task}_{run}.npz")
            
            if os.path.exists(filepath):
                try:
                    with np.load(filepath) as archive:
                        # We expect these to be shape (6, 20484)
                        sub_data_ts.append(archive['ts'])
                        sub_data_betas.append(archive['beta'])
                except Exception as e:
                    print(f"Error reading {filepath}: {e}")
                    files_missing = True
                    break
            else:
                print(f"Missing file for subject {sub}: {task}_{run}")
                files_missing = True
                break
                
        # Skip if incomplete data
        if files_missing or len(sub_data_ts) != 4:
            continue
            
        # 3. Compute the 4x4 correlation matrix for each condition
        # Initialize a dictionary to store this subject's results
        subject_results = {
            'subject_id': sub,
            'group': group_name,
            'ts': {},
            'beta': {}
        }
        
        for i, cond in enumerate(conditions):
            # Extract the i-th row (condition) across all 4 runs
            # This creates a list of 4 vectors, each of length 20484
            ts_cond_runs = [run_array[i, :] for run_array in sub_data_ts]
            betas_cond_runs = [run_array[i, :] for run_array in sub_data_betas]
            
            # Stack into a (4, 20484) matrix and compute the 4x4 correlation matrix
            corr_mat_ts = np.corrcoef(np.vstack(ts_cond_runs))
            corr_mat_betas = np.corrcoef(np.vstack(betas_cond_runs))
            
            # Save the 4x4 matrices into our dictionary
            subject_results['ts'][cond] = corr_mat_ts
            subject_results['beta'][cond] = corr_mat_betas
            
        # 4. Save the subject's dictionary to a .npy file
        output_filepath = os.path.join(output_dir, f"sub_{sub}_corr_matrices.npy")
        np.save(output_filepath, subject_results)
        print(f"Saved matrices for {group_name} subject {sub} -> {output_filepath}")

# --- EXECUTE ---
folder_path = './NEP/GLM/prosem' 
output_path = os.path.join(folder_path, 'prosem_mat')
compute_condition_correlation_matrices(folder_path, output_path)

Scanning folder: ./NEP/GLM/prosem
Saved matrices for native subject 3102 -> ./NEP/GLM/prosem/prosem_mat/sub_3102_corr_matrices.npy
Saved matrices for native subject 3103 -> ./NEP/GLM/prosem/prosem_mat/sub_3103_corr_matrices.npy
Saved matrices for native subject 3104 -> ./NEP/GLM/prosem/prosem_mat/sub_3104_corr_matrices.npy
Saved matrices for native subject 3105 -> ./NEP/GLM/prosem/prosem_mat/sub_3105_corr_matrices.npy
Saved matrices for native subject 3106 -> ./NEP/GLM/prosem/prosem_mat/sub_3106_corr_matrices.npy
Saved matrices for native subject 3107 -> ./NEP/GLM/prosem/prosem_mat/sub_3107_corr_matrices.npy
Saved matrices for native subject 3108 -> ./NEP/GLM/prosem/prosem_mat/sub_3108_corr_matrices.npy
Saved matrices for native subject 3109 -> ./NEP/GLM/prosem/prosem_mat/sub_3109_corr_matrices.npy
Saved matrices for native subject 3110 -> ./NEP/GLM/prosem/prosem_mat/sub_3110_corr_matrices.npy
Saved matrices for native subject 3111 -> ./NEP/GLM/prosem/prosem_mat/sub_3111_corr_matrices.

In [32]:
import os
import numpy as np
from scipy import stats
import pandas as pd

def run_similarity_ttests(input_dir):
    # 1. Define Groups and Conditions
    native = list(range(3102, 3117)) 
    learner = list(range(3202, 3217))
    subjects = native + learner
    
    conditions = [
        'positive_positive', 'positive_negative', 'positive_neutral', 
        'negative_negative', 'negative_positive', 'negative_neutral'
    ]
    
    # Store results for the final summary table
    results_summary = []

    print(f"Loading data from: {input_dir}\n")

    # 2. Iterate through data types (ts and betas)
    for data_type in ['ts', 'beta']:
        
        # 3. Iterate through each of the 6 conditions
        for cond in conditions:
            within_z_scores = []
            between_z_scores = []
            
            valid_subjects = 0
            
            # 4. Extract data for all 30 subjects
            for sub in subjects:
                filepath = os.path.join(input_dir, f"sub_{sub}_corr_matrices.npy")
                
                if not os.path.exists(filepath):
                    continue
                
                # Load the dictionary
                data = np.load(filepath, allow_pickle=True).item()
                
                # Get the 4x4 matrix for this specific data_type and condition
                mat = data[data_type][cond]
                
                # Extract Within-task r-values
                r_within_wd = mat[0, 1]
                r_within_sen = mat[2, 3]
                
                # Extract Between-task r-values
                r_between = [mat[0, 2], mat[0, 3], mat[1, 2], mat[1, 3]]
                
                # Apply Fisher Z-transform (np.arctanh) BEFORE averaging
                z_within_wd = np.arctanh(r_within_wd)
                z_within_sen = np.arctanh(r_within_sen)
                z_between_all = np.arctanh(r_between)
                
                # Average for this subject
                sub_mean_within = np.mean([z_within_wd, z_within_sen])
                sub_mean_between = np.mean(z_between_all)
                
                within_z_scores.append(sub_mean_within)
                between_z_scores.append(sub_mean_between)
                valid_subjects += 1
                
            # 5. Run the Paired T-Test across the N subjects
            if valid_subjects > 0:
                t_stat, p_val = stats.ttest_rel(within_z_scores, between_z_scores)
                
                # Calculate group means (optional: you can convert back to r using np.tanh for interpretation)
                mean_within = np.mean(within_z_scores)
                mean_between = np.mean(between_z_scores)
                
                results_summary.append({
                    'Data Type': data_type,
                    'Condition': cond,
                    'N Subjects': valid_subjects,
                    'Mean Z (Within)': round(mean_within, 3),
                    'Mean Z (Between)': round(mean_between, 3),
                    't-statistic': round(t_stat, 3),
                    'p-value': p_val
                })

    # 6. Display the Results
    df_results = pd.DataFrame(results_summary)
    
    # Add a column to easily flag significant results (p < 0.05)
    df_results['Significant'] = df_results['p-value'].apply(lambda p: '*' if p < 0.05 else '')
    
    # Format p-value for clean printing
    df_results['p-value'] = df_results['p-value'].apply(lambda p: f"{p:.4e}" if p < 0.001 else f"{p:.4f}")
    
    print(df_results.to_string(index=False))
    
    # Optional: Save results to a CSV
    csv_out = os.path.join(input_dir, 'similarity_ttest_results.csv')
    df_results.to_csv(csv_out, index=False)
    print(f"\nSaved statistical results to: {csv_out}")

# --- EXECUTE ---
folder_path = './NEP/GLM/prosem/prosem_mat' 
run_similarity_ttests(folder_path)

Loading data from: ./NEP/GLM/prosem/prosem_mat

Data Type         Condition  N Subjects  Mean Z (Within)  Mean Z (Between)  t-statistic p-value Significant
       ts positive_positive          30            0.587             0.586        0.059  0.9536            
       ts positive_negative          30            0.634             0.626        0.506  0.6167            
       ts  positive_neutral          30            0.622             0.613        0.441  0.6626            
       ts negative_negative          30            0.644             0.639        0.337  0.7387            
       ts negative_positive          30            0.640             0.633        0.372  0.7127            
       ts  negative_neutral          30            0.656             0.662       -0.395  0.6958            
     beta positive_positive          30            0.513             0.515       -0.145  0.8860            
     beta positive_negative          30            0.561             0.554        0.380 

In [37]:
import os
import numpy as np

def compute_condition_correlation_matrices(data_folder, output_dir='./subject_matrices'):
    """
    Loads (6, 20484) arrays for each run, computes a 4x4 correlation matrix 
    for each of the 6 conditions, and saves a .npy file per subject.
    """
    os.makedirs(output_dir, exist_ok=True)
    
    # 1. Define Groups
    native = list(range(3102, 3117)) 
    learner = list(range(3202, 3217))
    subjects = native + learner
    
    # 2. Define Conditions and Runs
    conditions = [
        'PP_vs_NP', 'Real_vs_Fake', 'PS_vs_NS', 'Congruent_vs_Incongruent', 'Congruent_vs_Neutral', 'Incongruent_vs_Neutral'
    ]
    
    run_configs = [
        ('wd', '01'), 
        ('wd', '02'), 
        ('sen', '01'), 
        ('sen', '02')
    ]
    
    print(f"Scanning folder: {data_folder}")
    
    for sub in subjects:
        # Determine group for metadata
        group_name = 'native' if sub in native else 'learner'
        
        sub_data_ts = []
        sub_data_betas = []
        files_missing = False
        
        # Load all 4 runs for this subject
        for task, run in run_configs:
            # Note: Assuming the files are still .npz as in the previous script.
            # If they are raw .npy files without 'ts'/'betas' keys, let me know!
            filepath = os.path.join(data_folder, f"{sub}_{task}_{run}.npz")
            
            if os.path.exists(filepath):
                try:
                    with np.load(filepath) as archive:
                        # We expect these to be shape (6, 20484)
                        sub_data_ts.append(archive['ts'])
                        sub_data_betas.append(archive['beta'])
                except Exception as e:
                    print(f"Error reading {filepath}: {e}")
                    files_missing = True
                    break
            else:
                print(f"Missing file for subject {sub}: {task}_{run}")
                files_missing = True
                break
                
        # Skip if incomplete data
        if files_missing or len(sub_data_ts) != 4:
            continue
            
        # 3. Compute the 4x4 correlation matrix for each condition
        # Initialize a dictionary to store this subject's results
        subject_results = {
            'subject_id': sub,
            'group': group_name,
            'ts': {},
            'beta': {}
        }
        
        for i, cond in enumerate(conditions):
            # Extract the i-th row (condition) across all 4 runs
            # This creates a list of 4 vectors, each of length 20484
            ts_cond_runs = [run_array[i, :] for run_array in sub_data_ts]
            betas_cond_runs = [run_array[i, :] for run_array in sub_data_betas]
            
            # Stack into a (4, 20484) matrix and compute the 4x4 correlation matrix
            corr_mat_ts = np.corrcoef(np.vstack(ts_cond_runs))
            corr_mat_betas = np.corrcoef(np.vstack(betas_cond_runs))
            
            # Save the 4x4 matrices into our dictionary
            subject_results['ts'][cond] = corr_mat_ts
            subject_results['beta'][cond] = corr_mat_betas
            
        # 4. Save the subject's dictionary to a .npy file
        output_filepath = os.path.join(output_dir, f"sub_{sub}_corr_matrices.npy")
        np.save(output_filepath, subject_results)
        print(f"Saved matrices for {group_name} subject {sub} -> {output_filepath}")

# --- EXECUTE ---
folder_path = './NEP/GLM/prosem_contrasts_revised' 
output_path = os.path.join(folder_path, 'prosem_contrasts_mat')
compute_condition_correlation_matrices(folder_path, output_path)

Scanning folder: ./NEP/GLM/prosem_contrasts_revised
Saved matrices for native subject 3102 -> ./NEP/GLM/prosem_contrasts_revised/prosem_contrasts_mat/sub_3102_corr_matrices.npy
Saved matrices for native subject 3103 -> ./NEP/GLM/prosem_contrasts_revised/prosem_contrasts_mat/sub_3103_corr_matrices.npy
Saved matrices for native subject 3104 -> ./NEP/GLM/prosem_contrasts_revised/prosem_contrasts_mat/sub_3104_corr_matrices.npy
Saved matrices for native subject 3105 -> ./NEP/GLM/prosem_contrasts_revised/prosem_contrasts_mat/sub_3105_corr_matrices.npy
Saved matrices for native subject 3106 -> ./NEP/GLM/prosem_contrasts_revised/prosem_contrasts_mat/sub_3106_corr_matrices.npy
Saved matrices for native subject 3107 -> ./NEP/GLM/prosem_contrasts_revised/prosem_contrasts_mat/sub_3107_corr_matrices.npy
Saved matrices for native subject 3108 -> ./NEP/GLM/prosem_contrasts_revised/prosem_contrasts_mat/sub_3108_corr_matrices.npy
Saved matrices for native subject 3109 -> ./NEP/GLM/prosem_contrasts_revi

In [2]:
import os
import numpy as np
import pandas as pd

def cronbach_alpha(X, rep_axis=0, var_axis=1, ci=None, squeeze=True):
    """Calculates Cronbach's alpha across the runs."""
    v1 = X.var(axis=var_axis, ddof=1, keepdims=True).sum(axis=rep_axis, keepdims=True)
    v2 = X.sum(axis=rep_axis, keepdims=True).var(axis=var_axis, ddof=1, keepdims=True)
    if squeeze:
        v1 = np.squeeze(v1, axis=(rep_axis, var_axis))
        v2 = np.squeeze(v2, axis=(rep_axis, var_axis))
    n = X.shape[rep_axis]
    
    if v2 == 0:
        return 0.0
        
    alpha = (n / (n - 1.0)) * (1.0 - v1 / v2)
    return float(alpha)

def compute_condition_reliability(data_folder, output_dir):
    """
    Loads beta arrays, computes overall Cronbach's alpha and 
    leave-one-out alpha for the 6 conditions, prints them, and saves to CSV.
    """
    os.makedirs(output_dir, exist_ok=True)
    
    native = list(range(3102, 3117)) 
    learner = list(range(3202, 3217))
    subjects = native + learner
    
    conditions = [
        'positive_positive', 'positive_negative', 'positive_neutral', 
        'negative_negative', 'negative_positive', 'negative_neutral'
    ]
    
    run_configs = [('wd', '01'), ('wd', '02'), ('sen', '01'), ('sen', '02')]
    run_names = [f"{t}_{r}" for t, r in run_configs]
    
    # This list will hold all our rows for the final CSV
    csv_data = []
    
    print(f"Scanning folder: {data_folder}")
    print("-" * 50)
    
    for sub in subjects:
        group_name = 'native' if sub in native else 'learner'
        sub_data_betas = []
        files_missing = False
        
        # Load the 4 runs
        for task, run in run_configs:
            filepath = os.path.join(data_folder, f"{sub}_{task}_{run}.npz")
            if os.path.exists(filepath):
                try:
                    with np.load(filepath) as archive:
                        sub_data_betas.append(archive['beta'])
                except Exception as e:
                    print(f"Error reading {filepath}: {e}")
                    files_missing = True
                    break
            else:
                files_missing = True
                break
                
        if files_missing or len(sub_data_betas) != 4:
            continue
            
        print(f"Subject {sub} ({group_name.upper()}):")
        
        for i, cond in enumerate(conditions):
            betas_cond_runs = np.vstack([run_array[i, :] for run_array in sub_data_betas])
            
            # 1. Calculate Overall Alpha
            overall_alpha = cronbach_alpha(betas_cond_runs)
            
            # Print to the cell
            print(f"  -> {cond}: \t Alpha = {overall_alpha:.3f}")
            
            # Prepare the row dictionary for the CSV
            row_dict = {
                'Subject': sub,
                'Group': group_name,
                'Condition': cond,
                'Overall_Alpha': overall_alpha
            }
            
            # 2. Calculate "Alpha if dropped"
            for run_idx in range(4):
                betas_dropped = np.delete(betas_cond_runs, run_idx, axis=0)
                alpha_drop = cronbach_alpha(betas_dropped)
                
                # Add the dropped alpha to our row dictionary
                col_name = f"Drop_{run_names[run_idx]}_Alpha"
                row_dict[col_name] = alpha_drop
                
            # Add this condition's data to our master list
            csv_data.append(row_dict)
            
        print("-" * 50)

    # Convert the list of dictionaries into a pandas DataFrame and save as CSV
    if csv_data:
        df = pd.DataFrame(csv_data)
        csv_filepath = os.path.join(output_dir, 'cronbach_alphas_summary.csv')
        df.to_csv(csv_filepath, index=False)
        print(f"\nSUCCESS: Saved comprehensive CSV summary to {csv_filepath}")
    else:
        print("\nNo data was processed. Please check your folder path and filenames.")

# --- EXECUTE ---
folder_path = './NEP/GLM/prosem' 
output_path = os.path.join(folder_path, 'prosem_mat')
compute_condition_reliability(folder_path, output_path)

Scanning folder: ./NEP/GLM/prosem
--------------------------------------------------
Subject 3102 (NATIVE):
  -> positive_positive: 	 Alpha = 0.766
  -> positive_negative: 	 Alpha = 0.811
  -> positive_neutral: 	 Alpha = 0.776
  -> negative_negative: 	 Alpha = 0.839
  -> negative_positive: 	 Alpha = 0.708
  -> negative_neutral: 	 Alpha = 0.818
--------------------------------------------------
Subject 3103 (NATIVE):
  -> positive_positive: 	 Alpha = 0.785
  -> positive_negative: 	 Alpha = 0.838
  -> positive_neutral: 	 Alpha = 0.759
  -> negative_negative: 	 Alpha = 0.778
  -> negative_positive: 	 Alpha = 0.813
  -> negative_neutral: 	 Alpha = 0.848
--------------------------------------------------
Subject 3104 (NATIVE):
  -> positive_positive: 	 Alpha = 0.762
  -> positive_negative: 	 Alpha = 0.861
  -> positive_neutral: 	 Alpha = 0.815
  -> negative_negative: 	 Alpha = 0.814
  -> negative_positive: 	 Alpha = 0.828
  -> negative_neutral: 	 Alpha = 0.765
------------------------------

In [41]:
import os
import numpy as np
from scipy import stats
import pandas as pd

def run_similarity_ttests(input_dir):
    # 1. Define Groups and Conditions
    native = list(range(3102, 3117)) 
    learner = list(range(3202, 3217))
    subjects = native + learner
    
    conditions = [
    'PP_vs_NP', 'Real_vs_Fake', 'PS_vs_NS', 'Congruent_vs_Incongruent', 'Congruent_vs_Neutral', 'Incongruent_vs_Neutral'
    ]
    
    # Store results for the final summary table
    results_summary = []

    print(f"Loading data from: {input_dir}\n")

    # 2. Iterate through data types (ts and betas)
    for data_type in ['ts', 'beta']:
        
        # 3. Iterate through each of the 6 conditions
        for cond in conditions:
            within_z_scores = []
            between_z_scores = []
            
            valid_subjects = 0
            
            # 4. Extract data for all 30 subjects
            for sub in subjects:
                filepath = os.path.join(input_dir, f"sub_{sub}_corr_matrices.npy")
                
                if not os.path.exists(filepath):
                    continue
                
                # Load the dictionary
                data = np.load(filepath, allow_pickle=True).item()
                
                # Get the 4x4 matrix for this specific data_type and condition
                mat = data[data_type][cond]
                
                # Extract Within-task r-values
                r_within_wd = mat[0, 1]
                r_within_sen = mat[2, 3]
                
                # Extract Between-task r-values
                r_between = [mat[0, 2], mat[0, 3], mat[1, 2], mat[1, 3]]
                
                # Apply Fisher Z-transform (np.arctanh) BEFORE averaging
                z_within_wd = np.arctanh(r_within_wd)
                z_within_sen = np.arctanh(r_within_sen)
                z_between_all = np.arctanh(r_between)
                
                # Average for this subject
                sub_mean_within = np.mean([z_within_wd, z_within_sen])
                sub_mean_between = np.mean(z_between_all)
                
                within_z_scores.append(sub_mean_within)
                between_z_scores.append(sub_mean_between)
                valid_subjects += 1
                
            # 5. Run the Paired T-Test across the N subjects
            if valid_subjects > 0:
                t_stat, p_val = stats.ttest_rel(within_z_scores, between_z_scores)
                
                # Calculate group means (optional: you can convert back to r using np.tanh for interpretation)
                mean_within = np.mean(within_z_scores)
                mean_between = np.mean(between_z_scores)
                
                results_summary.append({
                    'Data Type': data_type,
                    'Condition': cond,
                    'N Subjects': valid_subjects,
                    'Mean Z (Within)': round(mean_within, 3),
                    'Mean Z (Between)': round(mean_between, 3),
                    't-statistic': round(t_stat, 3),
                    'p-value': p_val
                })

    # 6. Display the Results
    df_results = pd.DataFrame(results_summary)
    
    # Add a column to easily flag significant results (p < 0.05)
    df_results['Significant'] = df_results['p-value'].apply(lambda p: '*' if p < 0.05 else '')
    
    # Format p-value for clean printing
    df_results['p-value'] = df_results['p-value'].apply(lambda p: f"{p:.4e}" if p < 0.001 else f"{p:.4f}")
    
    print(df_results.to_string(index=False))
    
    # Optional: Save results to a CSV
    csv_out = os.path.join(input_dir, 'similarity_ttest_results.csv')
    df_results.to_csv(csv_out, index=False)
    print(f"\nSaved statistical results to: {csv_out}")

# --- EXECUTE ---
folder_path = './NEP/GLM/prosem_contrasts_revised/prosem_contrasts_mat' 
run_similarity_ttests(folder_path)

Loading data from: ./NEP/GLM/prosem_contrasts_revised/prosem_contrasts_mat

Data Type                Condition  N Subjects  Mean Z (Within)  Mean Z (Between)  t-statistic p-value Significant
       ts                 PP_vs_NP          30            0.011             0.037       -1.144  0.2619            
       ts             Real_vs_Fake          30            0.081             0.097       -0.928  0.3611            
       ts                 PS_vs_NS          30            0.002             0.016       -0.579  0.5673            
       ts Congruent_vs_Incongruent          30            0.055             0.040        0.696  0.4919            
       ts     Congruent_vs_Neutral          30            0.059             0.089       -1.516  0.1402            
       ts   Incongruent_vs_Neutral          30            0.089             0.078        0.587  0.5614            
     beta                 PP_vs_NP          30            0.010             0.035       -1.136  0.2654            
    

In [8]:
def cronbach_alpha(X, rep_axis=0, var_axis=1, ci=None, squeeze=True):
    """Calculates Cronbach's alpha across the runs."""
    v1 = X.var(axis=var_axis, ddof=1, keepdims=True).sum(axis=rep_axis, keepdims=True)
    v2 = X.sum(axis=rep_axis, keepdims=True).var(axis=var_axis, ddof=1, keepdims=True)
    if squeeze:
        v1 = np.squeeze(v1, axis=(rep_axis, var_axis))
        v2 = np.squeeze(v2, axis=(rep_axis, var_axis))
    n = X.shape[rep_axis]
    
    if v2 == 0:
        return 0.0
        
    alpha = (n / (n - 1.0)) * (1.0 - v1 / v2)
    return float(alpha)

def compute_condition_reliability(data_folder, output_dir):
    """
    Loads beta arrays, computes overall Cronbach's alpha and 
    leave-one-out alpha for the 6 conditions, prints them, and saves to CSV.
    """
    os.makedirs(output_dir, exist_ok=True)
    
    native = list(range(3102, 3117)) 
    learner = list(range(3202, 3217))
    subjects = native + learner
    
    conditions = [
    'PP_vs_NP', 'Real_vs_Fake', 'PS_vs_NS', 'Congruent_vs_Incongruent', 'Congruent_vs_Neutral', 'Incongruent_vs_Neutral'
    ]
    
    run_configs = [('wd', '01'), ('wd', '02'), ('sen', '01'), ('sen', '02')]
    run_names = [f"{t}_{r}" for t, r in run_configs]
    
    # This list will hold all our rows for the final CSV
    csv_data = []
    
    print(f"Scanning folder: {data_folder}")
    print("-" * 50)
    
    for sub in subjects:
        group_name = 'native' if sub in native else 'learner'
        sub_data_betas = []
        files_missing = False
        
        # Load the 4 runs
        for task, run in run_configs:
            filepath = os.path.join(data_folder, f"{sub}_{task}_{run}.npz")
            if os.path.exists(filepath):
                try:
                    with np.load(filepath) as archive:
                        sub_data_betas.append(archive['beta'])
                except Exception as e:
                    print(f"Error reading {filepath}: {e}")
                    files_missing = True
                    break
            else:
                files_missing = True
                break
                
        if files_missing or len(sub_data_betas) != 4:
            continue
            
        print(f"Subject {sub} ({group_name.upper()}):")
        
        for i, cond in enumerate(conditions):
            betas_cond_runs = np.vstack([run_array[i, :] for run_array in sub_data_betas])
            
            # 1. Calculate Overall Alpha
            overall_alpha = cronbach_alpha(betas_cond_runs)
            
            # Print to the cell
            print(f"  -> {cond}: \t Alpha = {overall_alpha:.3f}")
            
            # Prepare the row dictionary for the CSV
            row_dict = {
                'Subject': sub,
                'Group': group_name,
                'Condition': cond,
                'Overall_Alpha': overall_alpha
            }
            
            # 2. Calculate "Alpha if dropped"
            for run_idx in range(4):
                betas_dropped = np.delete(betas_cond_runs, run_idx, axis=0)
                alpha_drop = cronbach_alpha(betas_dropped)
                
                # Add the dropped alpha to our row dictionary
                col_name = f"Drop_{run_names[run_idx]}_Alpha"
                row_dict[col_name] = alpha_drop
                
            # Add this condition's data to our master list
            csv_data.append(row_dict)
            
        print("-" * 50)

    # Convert the list of dictionaries into a pandas DataFrame and save as CSV
    if csv_data:
        df = pd.DataFrame(csv_data)
        csv_filepath = os.path.join(output_dir, 'cronbach_alphas_summary.csv')
        df.to_csv(csv_filepath, index=False)
        print(f"\nSUCCESS: Saved comprehensive CSV summary to {csv_filepath}")
    else:
        print("\nNo data was processed. Please check your folder path and filenames.")

# --- EXECUTE ---
folder_path = './NEP/GLM/prosem_contrasts_revised' 
output_path = os.path.join(folder_path, 'prosem_contrasts_mat')
compute_condition_reliability(folder_path, output_path)

Scanning folder: ./NEP/GLM/prosem_contrasts_revised
--------------------------------------------------
Subject 3102 (NATIVE):
  -> PP_vs_NP: 	 Alpha = 0.369
  -> Real_vs_Fake: 	 Alpha = -0.045
  -> PS_vs_NS: 	 Alpha = 0.179
  -> Congruent_vs_Incongruent: 	 Alpha = -0.069
  -> Congruent_vs_Neutral: 	 Alpha = 0.197
  -> Incongruent_vs_Neutral: 	 Alpha = -0.300
--------------------------------------------------
Subject 3103 (NATIVE):
  -> PP_vs_NP: 	 Alpha = 0.020
  -> Real_vs_Fake: 	 Alpha = 0.388
  -> PS_vs_NS: 	 Alpha = 0.085
  -> Congruent_vs_Incongruent: 	 Alpha = 0.080
  -> Congruent_vs_Neutral: 	 Alpha = 0.402
  -> Incongruent_vs_Neutral: 	 Alpha = 0.258
--------------------------------------------------
Subject 3104 (NATIVE):
  -> PP_vs_NP: 	 Alpha = 0.202
  -> Real_vs_Fake: 	 Alpha = 0.565
  -> PS_vs_NS: 	 Alpha = -0.541
  -> Congruent_vs_Incongruent: 	 Alpha = 0.359
  -> Congruent_vs_Neutral: 	 Alpha = 0.601
  -> Incongruent_vs_Neutral: 	 Alpha = 0.439
--------------------------

In [25]:
import os
import glob
import numpy as np
import pandas as pd

def get_best_runs_fisher_z(matrices_dir):
    """
    Finds the highest correlated pair of runs per subject by using a Fisher 
    z-transform to properly average the correlation matrices across contrasts.
    """
    search_pattern = os.path.join(matrices_dir, 'sub_*_corr_matrices.npy')
    subject_files = glob.glob(search_pattern)
    
    if not subject_files:
        print(f"No files found in {matrices_dir}.")
        return
        
    runs = ['wd_01', 'wd_02', 'sen_01', 'sen_02']
    
    # Your specific contrasts
    contrasts = [
        'PP_vs_NP', 'Real_vs_Fake', 'PS_vs_NS', 
        'Congruent_vs_Incongruent', 'Congruent_vs_Neutral', 'Incongruent_vs_Neutral'
    ]
    
    results_data = []

    print(f"Analyzing {len(subject_files)} subjects using Fisher z-transformed averages...\n")
    print("-" * 65)

    for filepath in sorted(subject_files):
        try:
            data = np.load(filepath, allow_pickle=True).item()
            sub_id = data['subject_id']
            group = data.get('group', 'unknown')
            
            ts_matrices = []
            beta_matrices = []
            
            for cond in contrasts:
                if cond in data['ts'] and cond in data['beta']:
                    # We use np.clip to prevent errors. The diagonal of a corr matrix 
                    # is 1.0, and arctanh(1.0) is infinity. Clipping to 0.9999 prevents this.
                    ts_clip = np.clip(data['ts'][cond], -0.9999, 0.9999)
                    beta_clip = np.clip(data['beta'][cond], -0.9999, 0.9999)
                    
                    # 1. Apply Fisher z-transform
                    ts_matrices.append(np.arctanh(ts_clip))
                    beta_matrices.append(np.arctanh(beta_clip))
                else:
                    print(f"Warning: Missing data for {cond} in {sub_id}")
                
            if not ts_matrices or not beta_matrices:
                continue
                
            # --- 2. Average the z-matrices & 3. Convert back to r (tanh) ---
            
            # For Time-Series (TS)
            mean_z_ts = np.mean(ts_matrices, axis=0)
            mean_r_ts = np.tanh(mean_z_ts) 
            
            upper_ts = np.triu(mean_r_ts, k=1)
            row_idx_ts, col_idx_ts = np.unravel_index(np.argmax(upper_ts), upper_ts.shape)
            ts_run_A, ts_run_B = runs[row_idx_ts], runs[col_idx_ts]
            ts_max_corr = upper_ts[row_idx_ts, col_idx_ts]
            
            # For Betas (Spatial Magnitude)
            mean_z_beta = np.mean(beta_matrices, axis=0)
            mean_r_beta = np.tanh(mean_z_beta)
            
            upper_beta = np.triu(mean_r_beta, k=1)
            row_idx_beta, col_idx_beta = np.unravel_index(np.argmax(upper_beta), upper_beta.shape)
            beta_run_A, beta_run_B = runs[row_idx_beta], runs[col_idx_beta]
            beta_max_corr = upper_beta[row_idx_beta, col_idx_beta]
            
            # --- 4. Save Results ---
            results_data.append({
                'Subject': sub_id,
                'Group': group.upper(),
                'TS_Best_Run_A': ts_run_A,
                'TS_Best_Run_B': ts_run_B,
                'TS_Max_Corr': round(float(ts_max_corr), 3),
                'Beta_Best_Run_A': beta_run_A,
                'Beta_Best_Run_B': beta_run_B,
                'Beta_Max_Corr': round(float(beta_max_corr), 3)
            })
            
            print(f"Sub {sub_id} | TS: {ts_run_A} & {ts_run_B} (r={ts_max_corr:.3f}) | Beta: {beta_run_A} & {beta_run_B} (r={beta_max_corr:.3f})")
            
        except Exception as e:
            print(f"Error processing {filepath}: {e}")

    print("-" * 65)
    
    if results_data:
        df = pd.DataFrame(results_data)
        csv_out_path = os.path.join(matrices_dir, 'best_runs_fisher_z_averaged.csv')
        df.to_csv(csv_out_path, index=False)
        print(f"SUCCESS: Saved Fisher-corrected CSV to -> {csv_out_path}")

# --- EXECUTE ---
matrix_folder = './NEP/GLM/prosem_contrasts_revised/prosem_contrasts_mat' 
get_best_runs_fisher_z(matrix_folder)

Analyzing 30 subjects using Fisher z-transformed averages...

-----------------------------------------------------------------
Sub 3102 | TS: wd_01 & wd_02 (r=0.121) | Beta: wd_01 & wd_02 (r=0.117)
Sub 3103 | TS: wd_01 & sen_01 (r=0.213) | Beta: wd_01 & sen_01 (r=0.203)
Sub 3104 | TS: wd_01 & sen_01 (r=0.249) | Beta: wd_01 & sen_01 (r=0.248)
Sub 3105 | TS: wd_01 & sen_01 (r=0.198) | Beta: wd_01 & sen_01 (r=0.194)
Sub 3106 | TS: wd_01 & sen_01 (r=0.122) | Beta: wd_01 & sen_01 (r=0.119)
Sub 3107 | TS: wd_02 & sen_01 (r=0.085) | Beta: wd_02 & sen_01 (r=0.082)
Sub 3108 | TS: wd_01 & sen_01 (r=0.148) | Beta: wd_01 & sen_01 (r=0.147)
Sub 3109 | TS: wd_02 & sen_02 (r=0.043) | Beta: wd_02 & sen_02 (r=0.042)
Sub 3110 | TS: wd_01 & wd_02 (r=0.050) | Beta: wd_01 & wd_02 (r=0.048)
Sub 3111 | TS: wd_01 & sen_01 (r=0.097) | Beta: wd_01 & sen_01 (r=0.097)
Sub 3112 | TS: wd_02 & sen_01 (r=0.047) | Beta: wd_02 & sen_01 (r=0.042)
Sub 3113 | TS: sen_01 & sen_02 (r=0.185) | Beta: sen_01 & sen_02 (r=0.181

In [30]:

import pandas as pd
from scipy import stats
import os

def run_group_ttest(csv_filepath):
    """
    Loads the highest correlated runs CSV and runs an independent samples t-test
    comparing the Beta_Max_Corr between NATIVE and LEARNER groups.
    """
    # 1. Load the data
    if not os.path.exists(csv_filepath):
        print(f"Error: Could not find {csv_filepath}")
        return
        
    df = pd.read_csv(csv_filepath)
    
    # Ensure Group column is uppercase to avoid matching errors
    df['Group'] = df['Group'].str.upper()
    
    # 2. Split the data into Native and Learner groups based on 'Beta_Max_Corr'
    native_data = df[df['Group'] == 'NATIVE']['TS_Max_Corr']
    learner_data = df[df['Group'] == 'LEARNER']['TS_Max_Corr']
    
    # Check if we successfully found data for both groups
    if native_data.empty or learner_data.empty:
        print("Error: Could not find data for both NATIVE and LEARNER groups.")
        return

    # 3. Calculate descriptive statistics (Means and Standard Deviations)
    native_mean = native_data.mean()
    native_std = native_data.std()
    
    learner_mean = learner_data.mean()
    learner_std = learner_data.std()
    
    # 4. Run the Independent Samples T-Test
    # Note: equal_var=False runs Welch's t-test, which is generally safer for fMRI data
    # as it does not assume both groups have the exact same variance in noise.
    t_stat, p_value = stats.ttest_ind(native_data, learner_data, equal_var=False)
    
    # 5. Print the Results
    print("="*50)
    print("       DATA RELIABILITY (BETA MAX CORR)        ")
    print("="*50)
    print(f"NATIVE  (n={len(native_data)}): Mean = {native_mean:.3f}, SD = {native_std:.3f}")
    print(f"LEARNER (n={len(learner_data)}): Mean = {learner_mean:.3f}, SD = {learner_std:.3f}")
    print("-" * 50)
    print(f"T-Statistic: {t_stat:.3f}")
    print(f"P-Value:     {p_value:.4f}")
    print("="*50)
    
    # Quick interpretation
    if p_value < 0.05:
        if native_mean > learner_mean:
            print("Conclusion: NATIVE speakers have significantly HIGHER reliability than LEARNERS.")
        else:
            print("Conclusion: LEARNERS have significantly HIGHER reliability than NATIVE speakers.")
    else:
        print("Conclusion: No significant difference in reliability between groups (p >= 0.05).")


# --- EXECUTE ---
csv_path = './NEP/GLM/prosem_contrasts_revised/prosem_contrasts_mat/highest_correlated_runs_ts_and_beta.csv'

run_group_ttest(csv_path)

       DATA RELIABILITY (BETA MAX CORR)        
NATIVE  (n=15): Mean = 0.141, SD = 0.078
LEARNER (n=15): Mean = 0.154, SD = 0.066
--------------------------------------------------
T-Statistic: -0.503
P-Value:     0.6187
Conclusion: No significant difference in reliability between groups (p >= 0.05).


In [3]:
import os
import glob
from itertools import combinations
import numpy as np
import pandas as pd


def get_all_pairwise_corrs_fisher_z(matrices_dir):
    """
    For every subject, computes the Fisher-z-averaged (across the 6 contrasts)
    correlation for ALL 6 possible run pairs -- not just the best one.

    This lets you see:
      - how correlated EVERY pair of runs is (not just the winning pair)
      - the average pairwise correlation per subject, which is the natural
        "typical 2-run reliability" number to compare against 3-run / 4-run
        alpha for a diminishing-returns check
    """
    search_pattern = os.path.join(matrices_dir, 'sub_*_corr_matrices.npy')
    subject_files = glob.glob(search_pattern)

    if not subject_files:
        print(f"No files found in {matrices_dir}.")
        return

    runs = ['wd_01', 'wd_02', 'sen_01', 'sen_02']
    pair_indices = list(combinations(range(4), 2))  # all 6 pairs

    contrasts = [
      'Real_vs_Fake'
    ]

    long_rows = []      # one row per subject x pair (TS and Beta corr)
    summary_rows = []   # one row per subject: mean / max pairwise corr, both TS and Beta

    for filepath in sorted(subject_files):
        try:
            data = np.load(filepath, allow_pickle=True).item()
            sub_id = data['subject_id']
            group = data.get('group', 'unknown')

            ts_matrices = []
            beta_matrices = []

            for cond in contrasts:
                if cond in data['ts'] and cond in data['beta']:
                    ts_clip = np.clip(data['ts'][cond], -0.9999, 0.9999)
                    beta_clip = np.clip(data['beta'][cond], -0.9999, 0.9999)
                    ts_matrices.append(np.arctanh(ts_clip))
                    beta_matrices.append(np.arctanh(beta_clip))

            if not ts_matrices or not beta_matrices:
                continue

            mean_r_ts = np.tanh(np.mean(ts_matrices, axis=0))
            mean_r_beta = np.tanh(np.mean(beta_matrices, axis=0))

            ts_pair_corrs = {}
            beta_pair_corrs = {}
            for (i, j) in pair_indices:
                pair_label = f"{runs[i]}_{runs[j]}"
                ts_pair_corrs[pair_label] = float(mean_r_ts[i, j])
                beta_pair_corrs[pair_label] = float(mean_r_beta[i, j])

                long_rows.append({
                    'Subject': sub_id,
                    'Group': group.upper(),
                    'Run_A': runs[i],
                    'Run_B': runs[j],
                    'TS_Corr': round(float(mean_r_ts[i, j]), 3),
                    'Beta_Corr': round(float(mean_r_beta[i, j]), 3),
                })

            ts_vals = list(ts_pair_corrs.values())
            beta_vals = list(beta_pair_corrs.values())

            summary_rows.append({
                'Subject': sub_id,
                'Group': group.upper(),
                'TS_Mean_Pair_Corr': round(float(np.mean(ts_vals)), 3),
                'TS_Max_Pair_Corr': round(float(np.max(ts_vals)), 3),
                'TS_Min_Pair_Corr': round(float(np.min(ts_vals)), 3),
                'Beta_Mean_Pair_Corr': round(float(np.mean(beta_vals)), 3),
                'Beta_Max_Pair_Corr': round(float(np.max(beta_vals)), 3),
                'Beta_Min_Pair_Corr': round(float(np.min(beta_vals)), 3),
            })

            print(f"Sub {sub_id}: TS mean/max/min = "
                  f"{np.mean(ts_vals):.3f}/{np.max(ts_vals):.3f}/{np.min(ts_vals):.3f} | "
                  f"Beta mean/max/min = {np.mean(beta_vals):.3f}/{np.max(beta_vals):.3f}/{np.min(beta_vals):.3f}")

        except Exception as e:
            print(f"Error processing {filepath}: {e}")

    if summary_rows:
        df_summary = pd.DataFrame(summary_rows)
        df_summary.to_csv(os.path.join(matrices_dir, 'all_pairwise_corrs_summary.csv'), index=False)
        print("Saved all_pairwise_corrs_summary.csv")

    if long_rows:
        df_long = pd.DataFrame(long_rows)
        df_long.to_csv(os.path.join(matrices_dir, 'all_pairwise_corrs_long.csv'), index=False)
        print("Saved all_pairwise_corrs_long.csv")


# --- EXECUTE ---
matrix_folder = './NEP/GLM/prosem_contrasts_revised/prosem_contrasts_mat'
get_all_pairwise_corrs_fisher_z(matrix_folder)

Sub 3102: TS mean/max/min = -0.007/0.074/-0.142 | Beta mean/max/min = -0.014/0.057/-0.147
Sub 3103: TS mean/max/min = 0.147/0.303/-0.001 | Beta mean/max/min = 0.142/0.283/0.002
Sub 3104: TS mean/max/min = 0.246/0.356/0.003 | Beta mean/max/min = 0.246/0.354/0.005
Sub 3105: TS mean/max/min = 0.032/0.405/-0.249 | Beta mean/max/min = 0.027/0.407/-0.250
Sub 3106: TS mean/max/min = 0.074/0.183/-0.099 | Beta mean/max/min = 0.071/0.178/-0.100
Sub 3107: TS mean/max/min = 0.007/0.103/-0.161 | Beta mean/max/min = 0.005/0.103/-0.160
Sub 3108: TS mean/max/min = 0.041/0.202/-0.050 | Beta mean/max/min = 0.037/0.203/-0.059
Sub 3109: TS mean/max/min = 0.011/0.135/-0.227 | Beta mean/max/min = 0.011/0.136/-0.221
Sub 3110: TS mean/max/min = 0.014/0.193/-0.131 | Beta mean/max/min = 0.014/0.190/-0.127
Sub 3111: TS mean/max/min = 0.035/0.101/-0.027 | Beta mean/max/min = 0.035/0.100/-0.029
Sub 3112: TS mean/max/min = 0.029/0.229/-0.104 | Beta mean/max/min = 0.017/0.218/-0.112
Sub 3113: TS mean/max/min = 0.107